# Comparaison des modèles APR — Campus d'Antananarivo
**Thèse** : Visualisation de SIG distribué sur des dispositifs mobiles

Ce notebook exécute :
1. L'entraînement de **MobileNet-APR-Campus C4** (notre modèle)
2. L'entraînement des **4 modèles de comparaison** (AlexNet, ResNet50V2, EfficientNetB0, MobileNetV1)
3. L'**évaluation comparative** avec métriques complètes (Haversine, MAE, taux de succès, inférence)
4. La génération du **tableau LaTeX** pour la section 2.13 de la thèse

## Structure attendue dans Google Drive
```
Mon Drive/
└── APR_Campus/
    ├── scripts/
    │   ├── train_mobilenet_apr_c4.py
    │   └── comparison_models.py
    ├── images/          ← images .jpg
    └── position_angle.csv
```

## Cellule 1 — Montage Drive et configuration

In [9]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# ── À adapter ──────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/test_doctorat/APR_Campus'
SCRIPTS_DIR  = os.path.join(DRIVE_ROOT, 'scripts')
IMAGES_DIR   = os.path.join(DRIVE_ROOT, 'images')
CSV_PATH     = os.path.join(DRIVE_ROOT, 'position_angle.csv')
OUTPUT_C4    = os.path.join(DRIVE_ROOT, 'resultats_comparison1')          # sorties entraînement C4
OUTPUT_CMP   = '/content/outputs_comparison'  # sorties comparaison
# ───────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_C4,  exist_ok=True)
os.makedirs(OUTPUT_CMP, exist_ok=True)

# Vérifications
checks = {'Scripts': SCRIPTS_DIR, 'Images': IMAGES_DIR, 'CSV': CSV_PATH}
all_ok = True
for label, path in checks.items():
    ok = os.path.exists(path)
    print(f"{'ok' if ok else 'x'}  {label:<10} : {path}")
    if not ok: all_ok = False

if all_ok:
    n_img = len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    import pandas as pd
    df_check = pd.read_csv(CSV_PATH, sep=',' if ',' in open(CSV_PATH).readline() else ';')
    print(f'\n {n_img} images dans {IMAGES_DIR}')
    print(f' {len(df_check)} lignes dans {os.path.basename(CSV_PATH)}')
    print(f'   Colonnes : {list(df_check.columns)}')
    print('\n Configuration valide — vous pouvez continuer.')
else:
    print('\n Corrigez les chemins ci-dessus avant de continuer.')

Mounted at /content/drive
ok  Scripts    : /content/drive/MyDrive/test_doctorat/APR_Campus/scripts
ok  Images     : /content/drive/MyDrive/test_doctorat/APR_Campus/images
ok  CSV        : /content/drive/MyDrive/test_doctorat/APR_Campus/position_angle.csv

 635 images dans /content/drive/MyDrive/test_doctorat/APR_Campus/images
 631 lignes dans position_angle.csv
   Colonnes : ['image_name', 'latitude', 'longitude', 'azimuth', 'pitch', 'roll']

 Configuration valide — vous pouvez continuer.


## Cellule 2 — GPU et dépendances

In [10]:
import subprocess, tensorflow as tf
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print(f" GPU : {r.stdout.strip() if r.returncode==0 else 'NON DÉTECTÉ'}")
print(f" TensorFlow : {tf.__version__}")
print(f"   GPUs TF  : {tf.config.list_physical_devices('GPU')}")
if not tf.config.list_physical_devices('GPU'):
    print('\n  Allez dans : Exécution → Modifier le type d\'exécution → GPU → T4')

 GPU : Tesla T4, 15360 MiB
 TensorFlow : 2.21.0
   GPUs TF  : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Cellule 3 — Ajout des scripts au path Python

In [11]:
import sys
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

# Test import
ok = True
for mod in ['train_mobilenet_apr_c4', 'comparison_models']:
    try:
        __import__(mod)
        print(f'   {mod}')
    except Exception as e:
        print(f'   {mod} — {e}')
        ok = False

if ok:
    print('\n Scripts prêts.')
else:
    print('\n Vérifiez que train_mobilenet_apr_c4.py et comparison_models.py sont dans', SCRIPTS_DIR)

   train_mobilenet_apr_c4
   comparison_models

 Scripts prêts.


---
# PARTIE A — Entraînement de MobileNet-APR-Campus C4
> **Architecture** : MobileNetV2 (ImageNet) + CBAM + Tête APR duale + Quaternion unitaire  
> **Protocole** : Phase 1 backbone gelé (10 ép.) → Phase 2 affinage global cosine decay (30 ép.)

## Cellule A1 — Paramètres d'entraînement C4

In [12]:
# ── Paramètres — modifiez ici ──────────────────────────────────────────────
C4_PHASE1_EP  = 10    # époques backbone gelé
C4_PHASE2_EP  = 30    # époques affinage global
C4_BATCH      = 32
C4_TEST_SIZE  = 0.20  # 20% pour le test (jamais utilisé pendant l'entraînement)
C4_VAL_SIZE   = 0.15  # 15% pour la validation (early stopping)
C4_SEED       = 42
# ───────────────────────────────────────────────────────────────────────────

print(f'  Phase 1 : {C4_PHASE1_EP} époques (backbone gelé)')
print(f'  Phase 2 : {C4_PHASE2_EP} époques (affinage global, cosine decay)')
print(f'  Split   : train={1-C4_TEST_SIZE-C4_VAL_SIZE:.0%} / val={C4_VAL_SIZE:.0%} / test={C4_TEST_SIZE:.0%}')
print(f'  Graine  : {C4_SEED}')

  Phase 1 : 10 époques (backbone gelé)
  Phase 2 : 30 époques (affinage global, cosine decay)
  Split   : train=65% / val=15% / test=20%
  Graine  : 42


## Cellule A2 — Chargement des données + rapport de reproductibilité

In [13]:
import train_mobilenet_apr_c4 as c4_module

data_c4 = c4_module.load_dataset(
    csv_path   = CSV_PATH,
    images_dir = IMAGES_DIR,
    test_size  = C4_TEST_SIZE,
    val_size   = C4_VAL_SIZE,
    seed       = C4_SEED,
)


  RAPPORT DE REPRODUCTIBILITÉ — Jeu de données
  Fichier CSV         : /content/drive/MyDrive/test_doctorat/APR_Campus/position_angle.csv
  Images totales      : 631
  Résolution native   : 4096 × 3072 px (12 MP utilisés pour la capture)
  Résolution utilisée : 1920 × 1440 px (format de stockage des images)
  Appareil            : Xiaomi Redmi Note 14
  Capteur caméra      : Samsung ISOCELL JN1  (1/2.76", 50 MP)
  Focale              : 5.58 mm (équivalent 24 mm plein format)
  Précision GPS       : ~3–5 m (GNSS intégré, conditions dégagées)
  Fréquence capture   : 2 Hz (intervalle 500 ms)
  Capteur orientation : SensorManager.getOrientation() — angles en radians

  Stratégie de split  : aléatoire (graine=42)
  Test                : 20% du total
  Validation          : 15% du total
  Train               : 65% du total
  Normalisation pos.  : min-max centrée (fit sur TRAIN uniquement)
  Normalisation img.  : statistiques ImageNet (µ=[0.485 0.456 0.406], σ=[0.229 0.224 0.225])

  ⚠ Limit

## Cellule A3 — Entraînement

In [15]:
model_c4, hist_c4 = c4_module.train(
    data_c4,
    batch     = C4_BATCH,
    phase1_ep = C4_PHASE1_EP,
    phase2_ep = C4_PHASE2_EP,
    output_dir= OUTPUT_C4,
)
print(f'\n Modèle sauvegardé dans {OUTPUT_C4}')

=== Phase 1 : backbone gelé ===
  Ép   2/10 train=0.3429(pos=0.3267,ori=0.4207) val=0.3112(pos=0.2996,ori=0.4092) sx=-0.026 sq=-0.026
  Ép   4/10 train=0.1707(pos=0.1839,ori=0.3196) val=0.1890(pos=0.1898,ori=0.3728) sx=-0.055 sq=-0.053
  Ép   6/10 train=0.0363(pos=0.1319,ori=0.2114) val=0.1139(pos=0.1738,ori=0.3379) sx=-0.083 sq=-0.081
  Ép   8/10 train=-0.0860(pos=0.0909,ori=0.1234) val=0.0441(pos=0.1609,ori=0.3122) sx=-0.111 sq=-0.109
  Ép  10/10 train=-0.1615(pos=0.0757,ori=0.0987) val=-0.0220(pos=0.1454,ori=0.2972) sx=-0.139 sq=-0.137

=== Phase 2 : affinage global (backbone dégelé) ===
  Ép   6/30 train=-0.2524(pos=0.0283,ori=0.0363) val=-0.0454(pos=0.1293,ori=0.2956)
  Ép  12/30 train=-0.2774(pos=0.0192,ori=0.0232) val=-0.0534(pos=0.1275,ori=0.3016)
  Ép  18/30 train=-0.2905(pos=0.0144,ori=0.0180) val=-0.0581(pos=0.1262,ori=0.3053)
  Ép  24/30 train=-0.2960(pos=0.0110,ori=0.0164) val=-0.0593(pos=0.1265,ori=0.3062)
  Ép  30/30 train=-0.2967(pos=0.0104,ori=0.0162) val=-0.0594(pos=0

# Conversion keras en tflite

In [17]:
import os
import numpy as np
import tensorflow as tf

# Scripts nécessaires pour les couches personnalisées
import sys
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
from train_mobilenet_apr_c4 import L2NormQuaternion, SpatialAttentionCBAM

# ── Chemins ────────────────────────────────────────────────────────────────
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, 'resultats_comparison1')
KERAS_PATH    = os.path.join(DRIVE_RESULTS, 'best_c4.keras')
TFLITE_FP32   = os.path.join(DRIVE_RESULTS, 'mobilenet_apr_c4_fp32.tflite')
TFLITE_FP16   = os.path.join(DRIVE_RESULTS, 'mobilenet_apr_c4_fp16.tflite')
# ───────────────────────────────────────────────────────────────────────────

# 1. Chargement du modèle Keras
model = tf.keras.models.load_model(
    KERAS_PATH,
    custom_objects={'L2NormQuaternion': L2NormQuaternion,
                    'SpatialAttentionCBAM': SpatialAttentionCBAM},
    compile=False,
)
print('Entrées  :', [(i.name, i.shape) for i in model.inputs])
print('Sorties  :', [(o.name, o.shape) for o in model.outputs])

def convert(model, out_path, fp16=False):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # Repli sur certains ops TF si un op n'a pas d'équivalent TFLite natif
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    if fp16:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    with open(out_path, 'wb') as f:
        f.write(tflite_model)
    print(f'{os.path.basename(out_path):<35} {os.path.getsize(out_path)/1024/1024:.2f} Mo')

# 2. Conversion float32 (précision maximale) et float16 (2x plus léger)
convert(model, TFLITE_FP32, fp16=False)
convert(model, TFLITE_FP16, fp16=True)

# 3. Vérification : comparaison Keras vs TFLite sur une entrée aléatoire
def run_tflite(path, x):
    interp = tf.lite.Interpreter(model_path=path)
    interp.allocate_tensors()
    inp  = interp.get_input_details()
    outs = interp.get_output_details()
    print(f'\n{os.path.basename(path)}')
    print('  input :', [(d["shape"].tolist(), d["dtype"].__name__) for d in inp])
    print('  output:', [(d["name"], d["shape"].tolist()) for d in outs])
    interp.set_tensor(inp[0]['index'], x)
    interp.invoke()
    return [interp.get_tensor(o['index']) for o in outs]

input_shape = tuple(model.inputs[0].shape[1:])
x = np.random.rand(1, *input_shape).astype(np.float32)

keras_out = model.predict(x, verbose=0)
keras_out = keras_out if isinstance(keras_out, (list, tuple)) else [keras_out]

for path in (TFLITE_FP32, TFLITE_FP16):
    tfl_out = run_tflite(path, x)
    for k, t in zip(sorted(keras_out, key=lambda a: a.shape[-1]),
                    sorted(tfl_out,   key=lambda a: a.shape[-1])):
        print(f'  écart max Keras/TFLite : {np.abs(k - t).max():.6f}')

print(f'\nFichiers .tflite enregistrés dans : {DRIVE_RESULTS}')

Entrées  : [('input_layer_2', (None, 224, 224, 3))]
Sorties  : [('keras_tensor_810', (None, 2)), ('keras_tensor_812', (None, 4))]
Saved artifact at '/tmp/tmpmxj2qvb0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  List[TensorSpec(shape=(None, 2), dtype=tf.float32, name=None), TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)]
Captures:
  137597229085136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229078416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229084752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229073232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229086096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229083408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229083792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137597229073424: TensorSpec(s

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Cellule A4 — Courbes d'entraînement

In [11]:
c4_module.plot_curves(hist_c4, C4_PHASE1_EP, C4_PHASE2_EP, OUTPUT_C4)

📊 Courbes → /content/outputs_c4/c4_training_curves.png


## Cellule A5 — Évaluation finale C4 avec distance Haversine

In [12]:
import numpy as np

metrics_c4 = c4_module.evaluate(model_c4, data_c4)
c4_module.plot_errors_per_image(metrics_c4['distances'], OUTPUT_C4)
c4_module.print_final_report(metrics_c4, data_c4)

📊 Erreurs par image → /content/outputs_c4/c4_per_image_errors.png

  RAPPORT D'ÉVALUATION FINALE — MobileNet-APR-Campus C4
  Jeu de test : 127 images
  (Total : 631 | Train : 409 | Val : 95)

  Formule Haversine :
  d = 2R·arcsin(√(sin²(Δφ/2) + cos(φ₁)·cos(φ₂)·sin²(Δλ/2)))
  R = 6 371 000 m

  MAE latitude    : 0.000108°
  MAE longitude   : 0.000136°
  MAE position    : 0.000245°
  Haversine min   : 1.05 m
  Haversine moy.  : 20.52 m
  Haversine méd.  : 18.45 m
  Haversine max   : 89.65 m
  Succès < 2 m    : 1.6%
  Succès < 10 m   : 22.0%
  Succès < 20 m   : 53.5%
  Ori. méd. (°)   : 34.509°


---
# PARTIE B — Comparaison avec les 4 autres modèles
> **Modèles** : AlexNet | ResNet50V2 | Sivapong CNN | MobileNetV1 | **MobileNet-APR-C4 ★**  
> Le **même split** (graine=42, 65/15/20%) est utilisé pour tous les modèles.  
> ⏱️ **Durée estimée GPU T4** : ~1–3 h selon le nombre d'époques.

## Cellule B1 — Paramètres de comparaison

In [13]:
# ── Paramètres — modifiez ici ──────────────────────────────────────────────
CMP_EPOCHS    = 50    # époques pour les 4 modèles de comparaison
CMP_BATCH     = 32
CMP_SEED      = 42    # DOIT être identique à C4_SEED pour un split équitable
# Chemin vers le modèle C4 déjà entraîné (Partie A)
# Si vide, le modèle sera réentraîné ici.
C4_MODEL_PATH = os.path.join(OUTPUT_C4, 'mobilenet_apr_c4_final.keras')
# ───────────────────────────────────────────────────────────────────────────

assert CMP_SEED == C4_SEED, ('!!! CMP_SEED doit être identique à C4_SEED '
                              'pour garantir le même split test pour tous les modèles !')
print(f'  Époques (comparaison) : {CMP_EPOCHS}')
print(f'  Graine                : {CMP_SEED}')
print(f'  Modèle C4             : {C4_MODEL_PATH}')
print(f'  Existe                : {os.path.exists(C4_MODEL_PATH)}')

  Époques (comparaison) : 50
  Graine                : 42
  Modèle C4             : /content/outputs_c4/mobilenet_apr_c4_final.keras
  Existe                : False


## Cellule B2 — Chargement des données (split identique pour tous)

In [14]:
import comparison_models as cmp

data_cmp = cmp.load_dataset(
    csv_path   = CSV_PATH,
    images_dir = IMAGES_DIR,
    test_size  = C4_TEST_SIZE,
    val_size   = C4_VAL_SIZE,
    seed       = CMP_SEED,
)
print(f"\n Données chargées — split identique à C4 (graine={CMP_SEED})")

Chargement de 631 images...

  RAPPORT DE REPRODUCTIBILITÉ — Jeu de données
  Images totales      : 631
  Train               : 409  (65%)
  Validation          : 95   (15%)
  Test                : 127  (20%)
  Graine              : 42
  Résolution utilisée : 1920 × 1440 px
  Appareil            : Xiaomi Redmi Note 14
  Précision GPS       : ~3–5 m
  Fréquence capture   : 2 Hz
  ⚠ Split aléatoire sans contrainte spatiale/temporelle


 Données chargées — split identique à C4 (graine=42)


## Cellule B3 — Entraînement des 4 modèles de comparaison

In [15]:
import numpy as np

# Angles en z-score pour les modèles baseline
ang_raw = None
try:
    import pandas as pd
    with open(CSV_PATH) as f: hdr = f.readline()
    sep = ',' if hdr.count(',') > hdr.count(';') else ';'
    df_raw = pd.read_csv(CSV_PATH, sep=sep)
    ang_raw = df_raw[['azimuth','pitch','roll']].values
    # Appliquer le même split
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(df_raw))
    trval, test_idx = train_test_split(idx, test_size=C4_TEST_SIZE, random_state=CMP_SEED)
    tr_idx, val_idx = train_test_split(trval, test_size=C4_VAL_SIZE/(1-C4_TEST_SIZE), random_state=CMP_SEED)
    mean_ang = ang_raw[tr_idx].mean(0)
    std_ang  = ang_raw[tr_idx].std(0)
    std_ang  = np.where(std_ang == 0, 1, std_ang)
    ang_tr_z  = ((ang_raw[tr_idx]  - mean_ang) / std_ang).astype(np.float32)
    ang_val_z = ((ang_raw[val_idx] - mean_ang) / std_ang).astype(np.float32)
except Exception as e:
    print(f'Angles non disponibles ({e}) — utilisation de zéros')
    ang_tr_z  = np.zeros((data_cmp['n_train'], 3), np.float32)
    ang_val_z = np.zeros((data_cmp['n_val'],   3), np.float32)

trained_models = {}

configs_baseline = [
    ('AlexNet',       cmp.build_alexnet,       data_cmp['X227_tr'], data_cmp['X227_val']),
    ('ResNet50V2',    cmp.build_resnet50,       data_cmp['X224_tr'], data_cmp['X224_val']),
    ('Sivapong CNN',   cmp.build_sivapong,      data_cmp['X224_tr'], data_cmp['X224_val']),
    ('MobileNetV1',   cmp.build_mobilenet_v1,   data_cmp['X224_tr'], data_cmp['X224_val']),
]

for name, builder, Xtr, Xval in configs_baseline:
    model = builder()
    cmp.train_baseline(
        model, Xtr, data_cmp['pos_z_tr'], ang_tr_z,
        Xval, data_cmp['pos_z_val'], ang_val_z,
        epochs=CMP_EPOCHS, batch=CMP_BATCH, model_name=name,
    )
    trained_models[name] = model
    print(f'- {name} entraîné')

print('\n Tous les modèles de comparaison entraînés.')


--- Entraînement : AlexNet ---
  Ép  10/50  train_MAE=0.41503  val_MAE=0.35345
  Ép  20/50  train_MAE=0.25362  val_MAE=0.18419
  Ép  30/50  train_MAE=0.23109  val_MAE=0.12889
  Ép  40/50  train_MAE=0.22641  val_MAE=0.14713
  Early stopping à l'époque 42
- AlexNet entraîné
94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- Entraînement : ResNet50V2 ---
  Ép  10/50  train_MAE=0.53571  val_MAE=0.48226
  Ép  20/50  train_MAE=0.41742  val_MAE=0.41415
  Ép  30/50  train_MAE=0.37060  val_MAE=0.37190
  Ép  40/50  train_MAE=0.34898  val_MAE=0.34930
  Ép  50/50  train_MAE=0.30831  val_MAE=0.33256
- ResNet50V2 entraîné

--- Entraînement : Sivapong CNN ---
  Ép  10/50  train_MAE=0.43865  val_MAE=0.32898
  Ép  20/50  train_MAE=0.40633  val_MAE=0.30786
  Ép  30/50  train_MAE=0.38657  val_MAE=0.35189
  Early stopping à l'époque 38
- Sivapong CNN entraîné
17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- Entraînement : MobileNetV1 ---
  Ép  10/50  train_MAE=0.48413  val_MAE=0.35993
  Ép  2

## Cellule B4 — Chargement / entraînement de MobileNet-APR-C4

In [16]:
import tensorflow as tf
from train_mobilenet_apr_c4 import L2NormQuaternion, SpatialAttentionCBAM

if os.path.exists(C4_MODEL_PATH):
    print(f'Chargement du modèle C4 depuis {C4_MODEL_PATH}...')
    c4_model = tf.keras.models.load_model(
        C4_MODEL_PATH,
        custom_objects={'L2NormQuaternion': L2NormQuaternion,
                        'SpatialAttentionCBAM': SpatialAttentionCBAM}
    )
    print(' Modèle C4 chargé.')
else:
    print('Modèle C4 non trouvé — réentraînement...')
    c4_model, _ = cmp.train_c4(data_cmp,
                                epochs_p1=10,
                                epochs_p2=CMP_EPOCHS,
                                batch=CMP_BATCH)
    print(' Modèle C4 entraîné.')

trained_models['MobileNet-APR-C4'] = c4_model

Modèle C4 non trouvé — réentraînement...

--- Entraînement : MobileNet-APR-C4 ---
  [Ph1] Ép   2/10  train=0.2779  val=0.2515
  [Ph1] Ép   4/10  train=0.1606  val=0.1650
  [Ph1] Ép   6/10  train=0.0406  val=0.0862
  [Ph1] Ép   8/10  train=-0.0704  val=0.0305
  [Ph1] Ép  10/10  train=-0.1485  val=-0.0190
  [Ph2] Ép  10/50  train=-0.2437  val=-0.0602
  [Ph2] Ép  20/50  train=-0.2810  val=-0.0822
  [Ph2] Ép  30/50  train=-0.3012  val=-0.0920
  [Ph2] Ép  40/50  train=-0.3083  val=-0.0957
  [Ph2] Ép  50/50  train=-0.3088  val=-0.0959
 Modèle C4 entraîné.


## Cellule B5 — Évaluation comparative complète

In [17]:
import time

results = {}
rows    = []

for name, model in trained_models.items():
    print(f'  Évaluation : {name}')
    if name == 'MobileNet-APR-C4':
        m = cmp.eval_c4(
            model,
            data_cmp['X224_te'], data_cmp['pos_mm_te'],
            data_cmp['pos_raw_te'], data_cmp['quat_te'],
            data_cmp['pos_mean_mm'], data_cmp['pos_range'],
        )
    else:
        Xte = data_cmp['X227_te'] if name == 'AlexNet' else data_cmp['X224_te']
        m   = cmp.eval_baseline(
            model, Xte, data_cmp['pos_z_te'],
            data_cmp['pos_raw_te'], data_cmp['quat_te'],
            data_cmp['pos_mean_z'], data_cmp['pos_std_z'],
        )

    n_params = int(sum(v.numpy().size for v in model.variables))
    size_mb  = cmp.model_size_mb(model)
    Xte_inf  = data_cmp['X227_te'] if name == 'AlexNet' else data_cmp['X224_te']
    infer    = cmp.inference_ms(model, Xte_inf)

    m['params']   = n_params
    m['size_mb']  = size_mb
    m['infer_ms'] = infer
    results[name] = m

    import pandas as pd
    rows.append({
        'Modèle':            name,
        'MAE pos. (°)':      m['mae_pos_deg'],
        'Hav. méd. (m)':     m['haversine_median_m'],
        'Succès < 2 m (%)':  m['success_2m'],
        'Succès < 10 m (%)': m['success_10m'],
        'Succès < 20 m (%)': m['success_20m'],
        'Ori. méd. (°)':     m['ori_median_deg'],
        'Params':            n_params,
        'Taille (Mo)':       size_mb,
        'Inférence (ms)':    infer,
    })

cmp.print_summary_table(rows, data_cmp)

  Évaluation : AlexNet
  Évaluation : ResNet50V2


  Évaluation : Sivapong CNN
  Évaluation : MobileNetV1
  Évaluation : MobileNet-APR-C4

  TABLEAU COMPARATIF — APR sur campus d'Antananarivo
  Jeu de test : 127 images (total 631 | train 409 | val 95)
Modèle                    MAE(°)  Hav.méd(m)  <2m(%)  <10m(%)  <20m(%)   Ori(°)      Params     Mo     ms
-----------------------------------------------------------------------------------------------
 AlexNet                0.000082       12.17     0.8     38.6     81.9        —  59,331,467 679.07  17.52
 ResNet50V2             0.000200       30.99     0.0      5.5     26.8        —  24,090,631  96.57 239.31
 Sivapong CNN           0.000212       32.57     0.0      4.7     20.5        —  36,299,961 415.49  16.36
 MobileNetV1            0.000142       21.94     0.0     11.8     42.5        —   3,492,551  15.65 116.77
★MobileNet-APR-C4       0.000117       15.33     0.8     26.8     64.6     37.2   2,915,496  11.69 180.42
★ = notre modèle proposé



## Cellule B6 — Visualisations

In [18]:
cmp.plot_haversine_bars(results, OUTPUT_CMP)
cmp.plot_cdf(results, OUTPUT_CMP)
cmp.plot_multi_metrics(results, OUTPUT_CMP)

📊 → /content/outputs_comparison/comparison_haversine.png
📊 → /content/outputs_comparison/comparison_cdf.png
📊 → /content/outputs_comparison/comparison_multi.png


## Cellule B7 — Export CSV et tableau LaTeX (section 2.13)

In [20]:
import pandas as pd
df_results = pd.DataFrame(rows)

# CSV
csv_out = os.path.join(OUTPUT_CMP, 'comparison_results.csv')
df_results.to_csv(csv_out, index=False, sep=';')
print(f'📄 CSV → {csv_out}')

# LaTeX
cmp.export_latex(df_results, OUTPUT_CMP, data_info={
    'n_total': data_cmp['n_total'],
    'device':  'Xiaomi Redmi Note 14',
})

# Affichage du tableau LaTeX
tex_path = os.path.join(OUTPUT_CMP, 'comparison_results.tex')
with open(tex_path) as f:
    print('\n' + '='*65)
    print('  TABLEAU LaTeX — à copier dans la section 2.13')
    print('='*65)
    print(f.read())

📄 CSV → /content/outputs_comparison/comparison_results.csv
📄 LaTeX → /content/outputs_comparison/comparison_results.tex

  TABLEAU LaTeX — à copier dans la section 2.13
\begin{table}[H]
\centering
\caption{Comparaison des modèles APR sur le campus d'Antananarivo (631 images, split 65/15/20\%, graine 42). $\downarrow$ : plus faible est meilleur ; $\uparrow$ : plus élevé est meilleur.}
\label{tab:comparison_models}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lccccccccc}
\toprule
\textbf{Modèle} & \textbf{MAE pos.~($^\circ$)}$\downarrow$ & \textbf{Hav. méd.~(m)}$\downarrow$ & \textbf{Succ.\,$<$2\,m~(\%)}$\uparrow$ & \textbf{Succ.\,$<$10\,m~(\%)}$\uparrow$ & \textbf{Succ.\,$<$20\,m~(\%)}$\uparrow$ & \textbf{Ori.~méd.~($^\circ$)}$\downarrow$ & \textbf{Params} & \textbf{Taille~(Mo)}$\downarrow$ & \textbf{Infér.~(ms)}$\downarrow$ \\
\midrule
AlexNet & 0.000082 & 12.17 & 0.8 & 38.6 & 81.9 & — & 59,331,467 & 679.07 & 17.52 \\
ResNet50V2 & 0.000200 & 30.99 & 0.0 & 5.5 & 26.8 & — & 24,090,631 & 9

---
# PARTIE C — Sauvegarde sur Google Drive

In [8]:
import shutil

DRIVE_RESULTS = os.path.join(DRIVE_ROOT, 'resultats_comparison1')
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Fichiers à sauvegarder
files = [
    (os.path.join(OUTPUT_C4,  'c4_training_curves.png'),   'c4_training_curves.png'),
    (os.path.join(OUTPUT_C4,  'c4_per_image_errors.png'),  'c4_per_image_errors.png'),
    (os.path.join(OUTPUT_C4,  'mobilenet_apr_c4_final.keras'), 'mobilenet_apr_c4_final.keras'),
    (os.path.join(OUTPUT_CMP, 'comparison_results.csv'),   'comparison_results.csv'),
    (os.path.join(OUTPUT_CMP, 'comparison_results.tex'),   'comparison_results.tex'),
    (os.path.join(OUTPUT_CMP, 'comparison_haversine.png'), 'comparison_haversine.png'),
    (os.path.join(OUTPUT_CMP, 'comparison_cdf.png'),       'comparison_cdf.png'),
    (os.path.join(OUTPUT_CMP, 'comparison_multi.png'),     'comparison_multi.png'),
]

print(f'Copie vers {DRIVE_RESULTS} ...')
for src, dst_name in files:
    dst = os.path.join(DRIVE_RESULTS, dst_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'   {dst_name:<45} ({os.path.getsize(dst)/1024:.1f} Ko)')
    else:
        print(f'    {dst_name:<45} (non trouvé)')

print(f'\n Sauvegarde terminée → {DRIVE_RESULTS}')

Copie vers /content/drive/MyDrive/test_doctorat/APR_Campus/resultats_comparison1 ...
    c4_training_curves.png                        (non trouvé)
    c4_per_image_errors.png                       (non trouvé)
    mobilenet_apr_c4_final.keras                  (non trouvé)
    comparison_results.csv                        (non trouvé)
    comparison_results.tex                        (non trouvé)
    comparison_haversine.png                      (non trouvé)
    comparison_cdf.png                            (non trouvé)
    comparison_multi.png                          (non trouvé)

 Sauvegarde terminée → /content/drive/MyDrive/test_doctorat/APR_Campus/resultats_comparison1


In [1]:
!ls -lh /content/outputs_c4/

ls: cannot access '/content/outputs_c4/': No such file or directory
